In [ ]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_probability as tfp 
import sklearn
import numpy as np
import pandas as pd
import sklearn.preprocessing
import functions
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from tslearn.metrics import dtw
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import datetime
from collections import Counter
from matplotlib.patches import Patch
from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import CubicSpline, interp1d, PchipInterpolator
from imblearn.over_sampling import SMOTE

In [3]:
X_full, y_full = functions.load_SGCC_dataset()

# STAD-GAN Basics
- GAN (Generative Adversarial Network)
  - 2 competing networks (generator & discriminator)
    - **Discriminator**: aim to distinguish between *real* and *synthetic* data
      - Given an input X, discriminator outputs a probability that the input is from the real data distribution
      - Gives feedback to the *generator* on how well the generator is doing in trying to trick the discriminator
    - **Generator**: generates synthetic samples, *with the sole purpose* to try to fool the discriminator.
    - End goal: To produce synthethic inputs that are 'indistinguishable' from actual data 
- STAD-GAN based on reconstruction error to detect anomalies. It is a GAN-AutoEncoder based framework
  - encoder --> decoder --> encoder structure
  - Self-training
    - A teacher GAN model encodes the MTS input into latent representations and is used as input into a DNN classifier to detect pseudolabels of normal / abnormal based on reconstruction error
    - Teacher GAN model assign these pseudolabels along with original MTS, **sort** the MTS based on anomaly scores and **select** 'top quartile' to train a new GAN model (student)
    - Iteratively, GAN model ability to capture abnormal patterns improves gradually. Each student trained becomes a teacher for the next iteration of STAD-GAN
      - Other than the first iteration of GAN training (as teacher), the students GAN model is trained with the refined dataset to minimize the classification loss on the pseudo-labels

# General STAD-GAN Framework:
-  $GAN_{E1}$ -> $GAN_{D}$ -> $GAN_{E2}$
-  2 Encoders $GAN_{E1}$ and $GAN_{E2}$ are used to separate X and X' further apart from each other in latent space representations ($Z_X$ and $Z'_{X'}$)
   -  Supposed X is abnormal, X' reconstructed will have large reconstruction error. The extra encoder will force the abnormal latent representation to be farther away from normal latent representation


$GAN_{E1}(X)$ = Z:
- Encodes MTS X into latent space as Z
- Adopt a stacked-LSTM model
  - LSTM network with backward length T to encodes components of $X_t$ sequentially into a latent vector
  - Last hidden layer of LSTM's last time step as output for $GAN_{E1}$, denoted as $Z_t = h_t$ where $h_t$ is the m-dimensional latent representation

Z:
- Latent representation of MTS X
- Learns the compact representation, bottleneck feature of X

$GAN_{D}(Z)$ = X'
- Decodes latent representations from Z into X'
- Decodes iteratively using single-layer LSTM to form the hidden layer state at each timestep, then stack all hidden layer states to form the reconstructed sample $X'_t$
- Training:
  - Minimizes the MSE loss (l2-norm) between X and X'

X'
- Reconstructed series of the original MTS X

$GAN_{E2}(X')$ = Z'
- Same structure as $GAN_{E1}$, but different parameters
- Encodes X' into another latent representation Z', through the same stacked-LSTM model as $GAN_{E1}(X)$
- Training:
  - Train together with $GAN_{E1}$ and $GAN_{D}$ to minimize MSE loss (l2-norm) between Z and Z'

Z'
- Same dimension latent space as Z

$D_Z$: Discriminator Z from Z'
- Aim to separate latent space Z as far as possible from Z' by distinguish between original latent representation and the reconstructed latent representation
- This pushes the generators $GAN_{E1}$ and $GAN_{E2}$ to produce Z and Z' that are as 'similar' as possible
- 2-layer DNN to classify whether Z and Z' are distinguishable from each other. Uses ReLU

$D_X$: Discriminator X from X'
- Aim to distinguish the generated MTS and the real MTS
- Pushes $GAN_{E1}$ and $GAN_{E2}$ to produce the synthetic X' looking as real as possible
- 2-layer DNN to classify whether X' is a real or fake sample of X. Uses ReLU
- Supposed X is noisy / anomalous, generator can still generate a normal MTS sample X'
  - $D_X$ amplifies the reconstruction error of abnormal data points to make them more separable from normal patterns.

Generators and Discriminator are both trained alternatively to minimize the overall losses.
- Minimizing overall discrimator loss == maximizing the distance between the inference between X and X', Z and Z'
- Minimizing the overall generator loss == minimizing the adversarial loss
- Adopt WGANs, scores how 'real' a sample is. The use of WGANs has the advantage of stability in the training process and is free of mode collapse


DNN classifier: take Z as input
- Z embeds the feature information of the original time series, DNN brings forward the result of anomaly score of the timeseries
- 3 dense layers with sigmoid activation, trains with pseudo-labeled data sample to reduce Binary Cross Entroy (BNE)

At initialization, with no possible pseudolabels in the beginning, we exclude the use of DNN classifier
- After convergence with adversarial loss / reconstruction loss, use reconstruction loss (L2-norm) between $X_t$ and $X'_t$ as the initial anomaly score along with the threshold to define the initial pseudolabels 

Post initialization:
- Choose high confidence samples to form the refined dataset, which a student GAN model takes as input. Keep on self training until the pseudo-labels prediction are not changed from the previous iteration
  - Sort MTS dataset by anomaly score, in descending order. Highest values indicates more anomalous, while lower values indicates less anomalous (more normal)
- Self-training goal: 
  - Minimize the reconstruction error of GAN and the classification error of DNN using the refined dataset along with its pseudo-labels
- After training a student model, use the student model to predict the pseudo-labels of the entire MTS X, and redo the whole high-confidence selection process to feed into the next student model

Once trained, the current student model is taken as the main STAD-GAN model, and can be used to predict on the unseen data.

# Default Parameters. 
Unless explicitly specified, the default parameter settings are as fol￾lows. 
1. The sliding window size (the lookback length of LSTM) is set to T = 5. 
2. The length of a latent vector is 10. 
3. The percentages of top abnormal and normal data points in the refined dataset are q = 2% and p = 50%. 
4. The weights of different loss functions are as follows: α1 = 0.8, α2 = 0.2, α3 = 0.1, γ1 = 0.1, and γ2 = 0.9


# During training,
Adopt the RMSprop optimizer with a weight decay of 0.01 to optimize the model, the default learning rate of the optimizing generator model is 0.0001, and the learning rate of the optimizing discriminator is 0.00001.

In [4]:
def cubic_spline_interpolate(series:pd.DataFrame):
    # We dont interpolate poor quality timeseries. Interpolating poor quality data result in synthetic generation of poor data, which causes overfitting
    if (series.isna().sum()/len(series) > 0.50) or series.iloc[:300].isna().all() or series.iloc[-300:].isna().all():
        return series
    
    mask = ~series.isna()
    value_known = series[mask].values
    index_known = series.index[mask]
    
    cs = CubicSpline(index_known, value_known, extrapolate=False)
    y_interpolate = np.clip(cs(series.index), 0, a_max=None)
    
    return pd.Series(y_interpolate, index=series.index)

In [5]:
def select_top_confidence(top_anomaly, top_normal, dataset:pd.DataFrame, anomaly_score:pd.Series):
    """
    Select top confident entries for training student class

    Args:
        top_anomaly (float): ratio of most confident anomalous entries
        top_normal (float): ratio of most confidence normal entries
        dataset (pd.DataFrame): MTS dataset
        anomaly_score (pd.Series): pseudo-anomaly scores of MTS dataset X

    Returns:
        selected_MTS_X, selected_MTS_y
    """
    sorted_scores = anomaly_score.sort_values(ascending=False)
    anomalies = sorted_scores.head(int(np.floor(top_anomaly*len(anomaly_score))))
    normals = sorted_scores.tail(int(np.floor(top_normal*len(anomaly_score))))
    selected_labels = pd.concat([anomalies, normals]).sample(frac=1, random_state=123)
    selected_dataset = dataset.loc[selected_labels.index]
    return selected_dataset, selected_labels

def preprocess(dataset:pd.DataFrame, normalize=False):
    if dataset.isnull().values.any():
        dataset = dataset.copy().apply(cubic_spline_interpolate, axis=1)
    dataset = dataset.dropna(axis=0)
    if normalize:
        normalizer = MinMaxScaler()
        dataset = pd.DataFrame(normalizer.fit_transform(dataset.T).T, columns=dataset.columns, index=dataset.index)
    return dataset


In [6]:
X_interpolated_normalized = preprocess(X_full, normalize=True)
X_interpolated_normalized

,2014-01-01,2014-01-02,2014-01-03,2014-01-04,2014-01-05,2014-01-06,2014-01-07,2014-01-08,2014-01-09,2014-01-10,...,2016-10-22,2016-10-23,2016-10-24,2016-10-25,2016-10-26,2016-10-27,2016-10-28,2016-10-29,2016-10-30,2016-10-31
0,0.000000,0.000000,0.000547,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.001093,0.045380,0.001093,0.003280,0.031711,0.048660,0.019136,0.020776,0.038272,0.013669
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.400045,0.536460,0.384339,0.321292,0.367287,0.590756,0.427642,0.389948,0.312766,0.317254
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.062476,0.123824,0.141638,0.168611,0.142266,...,0.000000,0.000000,0.000000,0.017940,0.027851,0.030235,0.043658,0.036256,0.023585,0.014553
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.487067,0.475842,0.511957,0.450708,0.378233,0.361152,0.470473,0.392875,0.427282,0.406784
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.216907,0.216014,0.245788,0.232836,0.284346,0.253976,0.247724,0.212739,0.211250,0.265588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25864,0.358931,0.386810,0.488717,0.500146,0.502329,0.492357,0.513175,0.469646,0.519362,0.443514,...,0.098704,0.031737,0.171932,0.221066,0.114063,0.120906,0.259936,0.232494,0.038506,0.124254
25866,0.738219,0.504158,0.475918,0.548683,0.368330,0.151247,0.217082,0.133056,0.128032,0.142585,...,0.336105,0.482328,0.372315,0.348579,0.381324,0.353430,0.536902,0.487179,0.462924,0.399342
25867,0.707428,0.659718,0.627275,0.624560,0.596961,0.631166,0.635423,0.648928,0.690252,0.673664,...,0.479595,0.507193,0.410085,0.409204,0.509909,0.494275,0.514533,0.430490,0.395332,0.439518
25868,0.027506,0.028640,0.030633,0.027861,0.025558,0.022248,0.031527,0.014251,0.023703,0.027036,...,0.023829,0.032306,0.021446,0.022683,0.017184,0.019842,0.022912,0.021629,0.029190,0.027953


# Dataset definition (MTS):

MTS dataset:

```math
x = \begin{bmatrix} 
    x_1, x_2, ... , x_N
    \end{bmatrix},
    \text{where } x_t \in \R^M (1 \leq t \leq N) \text{is a M-dimensional vector representing the M measurements at time node t}
``` 

```math
    x = \begin{bmatrix} 
        x_{1,1} &  x_{1,2} & ... & x_{1,N} \\
        x_{2,1} &  x_{2,2} & ... & x_{2,N} \\
        \vdots & \vdots & \ddots & \vdots \\
        x_{M,1} &  x_{M,2} & ... &x_{M,N} \\
        \end{bmatrix}
    \text{Supposed M = 1 for univariate timeseries} \longrightarrow 
    x = \begin{bmatrix} 
        x_{1} &  x_{2} & ... & x_{N} \\
    \end{bmatrix}
```

Supposed using a lookback window (sliding window) of length T, sub-time series are defined as:
```math
    X_t = \begin{bmatrix} 
        x_{t-T+1} &  x_{t-T+2} & ... & x_{t} \\
    \end{bmatrix},
    \text{where we basically capture the T time nodes preceeding timestep t}
```
``` math
    X_t = \begin{bmatrix} 
        x_{1, (t-T+1)} &  x_{1, (t-T+2)} & ... & x_{1, t} \\
        x_{2, (t-T+1)} &  x_{2, (t-T+2)} & ... & x_{2, t} \\
        \vdots & \vdots & \ddots & \vdots \\
        x_{M, (t-T+1)} &  x_{M, (t-T+2)} & ... &x_{M, t} \\
    \end{bmatrix},
```

then we can define our **MTS dataset** $\chi$ as
```math
    \chi = \{X_t | t = T, T+1, ..., N\} = \{X_T, X_{T+1}, ... , X_{N}\} = 
    \{ 
        \begin{bmatrix}
            x_{1, (1)} &  x_{1, (2)} & ... & x_{1, T} \\
            x_{2, (1)} &  x_{2, (2)} & ... & x_{2, T} \\
            \vdots & \vdots & \ddots & \vdots \\
            x_{M, (1)} &  x_{M, (2)} & ... &x_{M, T} \\
        \end{bmatrix},
        \begin{bmatrix}
            x_{1, (2)} &  x_{1, (3)} & ... & x_{1, T+1} \\
            x_{2, (2)} &  x_{2, (3)} & ... & x_{2, T+1} \\
            \vdots & \vdots & \ddots & \vdots \\
            x_{M, (2)} &  x_{M, (3)} & ... &x_{M, T+1} \\
        \end{bmatrix}, ...,
        \begin{bmatrix}
            x_{1, (N-T+1)} &  x_{1, (N-T+2)} & ... & x_{1, N} \\
            x_{2, (N-T+1)} &  x_{2, (N-T+2)} & ... & x_{2, N} \\
            \vdots & \vdots & \ddots & \vdots \\
            x_{M, (N-T+1)} &  x_{M, (N-T+2)} & ... &x_{M, N} \\
        \end{bmatrix}
    \}
```

If univariate, then $\chi$
```math
    \chi = \{X_t | t = T, T+1, ..., N\} = \{X_T, X_{T+1}, ... , X_{N}\} = 
    \{ 
        \begin{bmatrix}
            x_1 &  x_2 & ... & x_T \\
        \end{bmatrix},
        \begin{bmatrix}
            x_2 &  x_3 & ... & x_{T+1} \\
        \end{bmatrix}, ...,
        \begin{bmatrix}
            x_{N-T+1} &  x_{N-T+2} & ... & x_{N} \\
        \end{bmatrix}
    \}
```

The goal for the anomaly detection is then to learn an anomaly score for each sample $X_t$ among MTS $\chi$, and form a classifier to predict whether an anomaly is detected at time t


# Summary
1. Given a timeseries, partition them into overlapping windows of length T
   1. Final shape (batch size (users), N-5+1, 5, 1)
2. LSTM encodes the window at a time t through stacked LSTM, where on intermediate layers it updates internal parameters to keep important patterns / details of the time window, and return the final state at last hidden layer of the last timestep as the encoded latent representation.
   1. For SGCC dataset each user maintain their own tensors of sliding windows, after the encoding, still gets stored sequentially as latent vectors (this keeps track of which window belong to which user)
   2. This is for full timeseries reconstruction purposes
3. Latent vector decoding consists of a single-layer LSTM to decode individual sliding windows
4. Generator-Discriminator trains simultaneously within STAD-GAN model
5. After convergence, DNN classifier trains by shuffling the windows (hence intermingling in the different users windows), and trains on latent vectors of corresponding window
6. Once anomaly scores are assigned for all windows, select top anomalies and top normals windows to further train student STAD-GAN model
7. Repeat until convergence
8. After convergence, use the STAD-GAN to process a user's timeseries and produces a score for every sliding windows, from which it labels the windows are normal or anomalous based on whether the score exceeds predefined threshold. 
   1. For SGCC: a user is defined as anomalous if they exhibit ANY anomalous data points (point anomaly), therefore if a user have any 1-window, we label the entire series as anomalous


In [7]:
class stacked_lstm_encoder(keras.layers.Layer):
    def __init__(self, input_shape=[7, 1], vec_size = 10, **kwargs):
        super().__init__(trainable=True, **kwargs)
        self.input_shape = input_shape
        self.latent_output_shape = vec_size
        
        if self.input_shape[1] == 1: # Univariate
            self.lstm1 = keras.layers.LSTM(64, return_sequences=True)
            self.lstm2 = keras.layers.LSTM(self.latent_output_shape, return_sequences=False)
        
        else: # Multivariate Time Series (MTS) 
            pass

    def call(self, inputs):
        # (number of users (batch), number of windows, windows length)
        assert len(inputs.shape) == 3, "Mismatching inputs shape"
        
        # Assume Univariate Timeseries
        B = tf.shape(inputs)[0]
        N_timestep = tf.shape(inputs)[1]
        x = tf.reshape(inputs, [-1, tf.shape(inputs)[2], 1])
        hidden = self.lstm1(x)
        latent_reps = self.lstm2(hidden)
        final_out = tf.reshape(latent_reps, [B, N_timestep, self.latent_output_shape])
        return final_out

flatten_latents:
```math
    \begin{bmatrix} 
        latent1_t, latent1_{t+1}, latent1_{t+2}, ..., latent1_{N} \\
        latent2_t, latent2_{t+1}, latent2_{t+2}, ..., latent2_{N} \\
        \vdots & \vdots \\
        latentB_t, latentB_{t+1}, latentB_{t+2}, ..., latentB_{N} \\
    \end{bmatrix}=
    \begin{bmatrix}
        user1 \text{ sliding windows latent representations} \\
        user2 \text{ sliding windows latent representations}\\
        \vdots \\
        userB \text{ sliding windows latent representations}
    \end{bmatrix}
```

In [8]:
class lstm_decoder(keras.layers.Layer):
    def __init__(self, out_length=7, **kwargs):
        super().__init__(**kwargs)

        self.out_length = out_length
        self.initial_h = keras.layers.Dense(64, activation='tanh')
        self.initial_c = keras.layers.Dense(64, activation='tanh')
        self.decoder = keras.layers.LSTM(64, return_sequences=True)
        self.out = keras.layers.TimeDistributed(keras.layers.Dense(1)) # Apply a dense layer compressing the each output of the decoder to a specified window size.
    
    def call(self, latent_reps):
        # expecting (batch, N_timesteps, latent_dim) --> output (batch, num_windows, features per time node)
        N_timesteps = tf.shape(latent_reps)[1]
        B = tf.shape(latent_reps)[0]
        
        # (B, N, 10) --> (B*N, 10)
        flatten_latents = tf.reshape(latent_reps, [-1, tf.shape(latent_reps)[-1]])
        
        # Initial states for h_0 (hidden), c_0 (cell state), used to iteratively decode information. Each state is represented by 64 length vector
        h_o = self.initial_h(flatten_latents)
        c_o = self.initial_c(flatten_latents)
        
        # (B*N, out_length, 1) sliding windows 
        placeholder = tf.zeros((tf.shape(flatten_latents)[0], self.out_length, 1))
        
        # projects the initial states onto the placeholder tensor
        # Produces (B*N, out_length, 64) where each time step within a timewindow is represented by the hidden-state vectors of size 64
        decoded = self.decoder(placeholder, initial_state=[h_o, c_o])
        
        # Passes the tensor of hidden states through the TimeDistributed Dense layer to map the 64-vector down to a observed load that was captured by the vector
        flatten_decoded = self.out(decoded)
        
        final_out = tf.reshape(flatten_decoded, [B, self.out_length, self.out_length])
        return self.out(final_out)

In [ ]:
class STAD_GAN_generator(keras.models.Model):
    def __init__(self, input_size=[7,1], latent_size=10, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.input_shape = input_size
        self.latent_size=latent_size
        self.encoder1 = stacked_lstm_encoder(input_shape=input_size, vec_size=self.latent_size)
        self.decoder = lstm_decoder(out_length=input_size[0])
        self.encoder2 = stacked_lstm_encoder(input_shape=input_size, vec_size=self.latent_size)
        self.optimizer = keras.optimizers.RMSprop(0.0001)
        
    def compile(self,optimizer=None, **kwargs):
        super().compile(**kwargs)
        if optimizer:
            self.optimizer = optimizer
            
    def train_step(self, data):
        """
        Expecting data = [Batch, number of windows, window size, 1 (univariate)]
        """
        
        # Keeps a record of operations and their dependencies on the trainable variables
        with tf.GradientTape() as tape:
            # Encode data into Z --> Enc1(data) = Z *************** This marks that 'Z' is produced through encoder1 using 'data', and hence backpropagation on Z would also update weights on encoder1
            Z = self.encoder1(data, training=True)
            
            # Decode data X_reconstructed --> Dec(Z) = X_reconstructed ******************* This marks that X_reconstructed is produced through decoder using Z, hence backpropagation on X_reconstructed would update
            # weights on decoder, and also on encoder1 by backpropagation. X_reconstructed dependent on both encoder1 and decoder
            X_reconstructed = self.decoder(Z, training=True)
            
            # Z_prime is therefore dependent on encoder2, decoder, and encoder1. Therefore encoder2 is trained in tandem with encoder1 and decoder
            Z_prime = self.encoder2(X_reconstructed, training=True)
            
            loss_decoder = self.loss(data, X_reconstructed)
            loss_encoder = self.loss(Z, Z_prime)
            overall_loss = loss_decoder + loss_encoder
        
        weights = self.encoder1.trainable_weights + self.decoder.trainable_weights + self.encoder2.trainable_weights
        gradient = tape.gradient(overall_loss, weights)
        self.optimizer.apply_gradients(zip(gradient, weights))
        return {
            "Overall_Loss" : overall_loss,
            "Decoder_loss" : loss_decoder,
            "Encoder_loss" : loss_encoder
        }
            

In [ ]:
class STAD_GAN_discriminator(keras.models.Model):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.model = keras.Sequential([
            keras.layers.Flatten(),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(1, activation='sigmoid'),
        ])
        self.optimizer = keras.optimizers.RMSprop(0.00001)

    def call(self, x, training=False):
        return self.model(x, training=training)

In [ ]:
class STAD_GAN_framework:
    def __init__(self, X_train:pd.DataFrame, num_features=1, latent_vec_size=10, frame_length=5, losses=[0.8, 0.2, 0.1, 0.1, 0.9]):
        self.teacher = None
        self.student = None
        self.frame_length = frame_length
        self.X_train = X_train
        self.num_features = num_features
        self.curr_latents = None
        self.bce = keras.losses.BinaryCrossentropy()
        self.mse = keras.losses.MeanSquaredError()
        self.curr_latents_reconstructed = None
        self.latent_size = latent_vec_size
        self.alpha1, self.alpha2, self.alpha3, self.gamma1, self.gamma2 = losses
        
    def train(self, X_train:pd.DataFrame, pseudolabels=None, epochs=20):
        """
        Train the STAD_GAN framework until full convergence. This method expect X_train to be univariate timeseries dataset of multiple users
            - Each row correspond to a user's timeseries
            - Each feature is univariate

        Args:
            X_train (pd.DataFrame): MTS dataframe, raw dataset of user's load
            pseudolabels (pd.Series): anomaly score of X_train samples (Higher = more anomalous)
        """
        assert len(X_train.shape) == 2, "Wrong dataset size"
        
        if self.teacher == None: # First iteration
            self.teacher = STAD_GAN_generator(input_size=[self.frame_length, self.num_features], latent_size=self.latent_size)
            self.disc_Z = STAD_GAN_discriminator()
            self.disc_X = STAD_GAN_discriminator()
            self.dnn_classifier = keras.models.Sequential([
                keras.layers.Dense(64, activation='sigmoid'),
                keras.layers.Dense(32, activation='sigmoid'),
                keras.layers.Dense(16, activation='sigmoid'),
                keras.layers.Dense(1, activation='sigmoid')
            ])
            
            self.user_timewindows = tf.signal.frame(X_train, frame_length=self.frame_length, frame_step=1)
            dataset = tf.data.Dataset.from_tensor_slices(self.user_timewindows).shuffle(1000).batch(256)
            
            for epoch in range(epochs):
                print(f"Epoch: {epoch+1}/{epochs}")
                for step, batch in enumerate(dataset):
                    with tf.GradientTape() as discriminator_step:
                        Z = self.teacher.encoder1(batch, training=False)
                        X_reconstructed = self.teacher.decoder(Z, training=False)
                        Z_prime = self.teacher.encoder2(X_reconstructed, training=False)
                        
                        real_X = self.disc_X(batch, training=True)
                        fake_X = self.disc_X(X_reconstructed, training=True)
                        
                        real_Z = self.disc_Z(Z, training=True)
                        fake_Z = self.disc_Z(Z_prime, training=True)
                        
                        disc_loss = -self.mse(real_X, fake_X) - self.mse(real_Z, fake_Z)
                    disc_gradient = discriminator_step.gradient(disc_loss, self.disc_X.trainable_weights+self.disc_Z.trainable_weights)
                    self.disc_X.optimizer.apply_gradients(zip(disc_gradient, self.disc_X.trainable_weights +self.disc_Z.trainable_weights))
                    
                    with tf.GradientTape() as generator_step:
                        Z = self.teacher.encoder1(batch, training=True)
                        X_reconstructed = self.teacher.decoder(Z, training=True)
                        Z_prime = self.teacher.encoder2(X_reconstructed, training=False)

                        decoder_loss = self.mse(batch, X_reconstructed)
                        encoder_loss = self.mse(Z, Z_prime)
                        generator_loss = self.alpha1*decoder_loss + self.alpha2*encoder_loss - self.alpha3*disc_loss
                    gen_weights = self.teacher.encoder1.trainable_weights + self.teacher.decoder.trainable_weights + self.teacher.encoder2.trainable_weights
                    generator_gradient = generator_step.gradient(generator_loss, gen_weights)
                    self.teacher.optimizer.apply_gradients(zip(generator_gradient, gen_weights))
                    
            # Initial teacher model uses reconstruction error to generate the initial pseudolabels
            reconstructed = []
            for _, batch in enumerate(dataset):
                Z = self.teacher.encoder1(batch, training=False)
                reconstructed.append(self.decoder(Z, training=False))
            
            X_reconstructed = tf.concat(reconstructed, axis=0)
            errors = tf.norm(self.user_timewindows - X_reconstructed, ord='euclidean', axis=[2,3])
            threshold = tfp.stats.percentile(errors, 95)
            pseudolabels = tf.cast(errors > threshold)
                        

In [12]:
windows = tf.signal.frame(X_interpolated_normalized, frame_length=5, frame_step=1, axis=1)

In [13]:
windows

<tf.Tensor: shape=(24204, 1030, 5), dtype=float64, numpy=
array([[[0.00000000e+00, 0.00000000e+00, 5.46746856e-04, 0.00000000e+00,
         0.00000000e+00],
        [0.00000000e+00, 5.46746856e-04, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00],
        [5.46746856e-04, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00],
        ...,
        [3.28048114e-03, 3.17113177e-02, 4.86604702e-02, 1.91361400e-02,
         2.07763805e-02],
        [3.17113177e-02, 4.86604702e-02, 1.91361400e-02, 2.07763805e-02,
         3.82722799e-02],
        [4.86604702e-02, 1.91361400e-02, 2.07763805e-02, 3.82722799e-02,
         1.36686714e-02]],

       [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
         0.00000000e+00],
        ...,
        [3.21292349e-01